In [1]:
# =========================================================
# 1. INSTALLATIONS
# =========================================================

!pip install -q transformers sentencepiece accelerate

In [2]:
# =========================================================
# 2. IMPORTS
# =========================================================

import os
import random
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.model_selection import train_test_split

from transformers import (
    RobertaTokenizer,
    RobertaModel
)

warnings.filterwarnings("ignore")

In [3]:
# =========================================================
# 3. CONFIG
# =========================================================

CONFIG = {

    "SEED": 42,

    "MAX_LENGTH": 16,

    "BATCH_SIZE": 32,

    "ROBERTA_MODEL": "roberta-base",

    "EMBEDDING_DIM": 256
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)

DEVICE: cpu


In [4]:
# =========================================================
# 4. SET SEED
# =========================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])

In [5]:
# ============================================================
# LOAD TESS DATASET
# ============================================================

DATASET_PATH = "/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data"

print(os.listdir(DATASET_PATH))

['YAF_fear', 'OAF_angry', 'OAF_Fear', 'OAF_disgust', 'OAF_neutral', 'YAF_angry', 'OAF_Sad', 'YAF_disgust', 'YAF_neutral', 'OAF_Pleasant_surprise', 'YAF_happy', 'OAF_happy', 'YAF_sad', 'YAF_pleasant_surprised']


In [6]:
# =========================================================
# 6. BUILD TEXT DATAFRAME
# =========================================================

data = []

emotion_map = {

    "angry": "angry",

    "disgust": "disgust",

    "fear": "fear",

    "happy": "happy",

    "neutral": "neutral",

    "ps": "pleasant_surprise",

    "sad": "sad"
}

for folder in os.listdir(DATASET_PATH):

    folder_path = os.path.join(
        DATASET_PATH,
        folder
    )

    for file in os.listdir(folder_path):

        if file.endswith(".wav"):

            parts = file.replace(".wav", "").split("_")

            # FORMAT:
            # OAF_word_emotion.wav

            word = parts[1]

            emotion = parts[2]

            transcript = f"Say the word {word}"

            data.append({

                "text": transcript,

                "emotion": emotion_map[emotion]
            })

df = pd.DataFrame(data)

print(df.head())

                  text emotion
0    Say the word home    fear
1   Say the word youth    fear
2    Say the word near    fear
3  Say the word search    fear
4    Say the word pick    fear


In [7]:
# =========================================================
# 7. LABEL ENCODING
# =========================================================

label2id = {

    label: idx

    for idx, label in enumerate(
        sorted(df["emotion"].unique())
    )
}

id2label = {

    v:k for k,v in label2id.items()
}

df["label"] = df["emotion"].map(label2id)

print(label2id)

{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'pleasant_surprise': 5, 'sad': 6}


In [8]:
# =========================================================
# 8. TRAIN TEST SPLIT
# =========================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.2,

    stratify=df["label"],

    random_state=CONFIG["SEED"]
)

print("Train Size:", len(train_df))

print("Test Size:", len(test_df))

Train Size: 2240
Test Size: 560


In [9]:
# =========================================================
# 9. TOKENIZER
# =========================================================

tokenizer = RobertaTokenizer.from_pretrained(
    CONFIG["ROBERTA_MODEL"]
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
# =========================================================
# 10. DATASET CLASS
# =========================================================

class TextDataset(Dataset):

    def __init__(self, dataframe):

        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        encoding = tokenizer(

            row["text"],

            padding="max_length",

            truncation=True,

            max_length=CONFIG["MAX_LENGTH"],

            return_tensors="pt"
        )

        return {

            "input_ids":
            encoding["input_ids"].squeeze(0),

            "attention_mask":
            encoding["attention_mask"].squeeze(0),

            "label":
            torch.tensor(row["label"])
        }

In [11]:
# =========================================================
# 11. DATALOADERS
# =========================================================

train_dataset = TextDataset(train_df)

test_dataset = TextDataset(test_df)

train_loader = DataLoader(

    train_dataset,

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False
)

test_loader = DataLoader(

    test_dataset,

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False
)

## RoBERTa EMOTION ENCODER ARCHITECTURE

In [12]:
# =========================================================
# 12. TEXT ENCODER
# =========================================================

class TextEncoder(nn.Module):

    def __init__(self):

        super(TextEncoder, self).__init__()

        # =================================================
        # LOAD PRETRAINED ROBERTA
        # =================================================

        self.roberta = RobertaModel.from_pretrained(
            CONFIG["ROBERTA_MODEL"]
        )

        # =================================================
        # PARTIAL UNFREEZING
        # =================================================

        for name, param in self.roberta.named_parameters():

            # FREEZE EVERYTHING FIRST
            param.requires_grad = False

            # UNFREEZE TOP 4 TRANSFORMER LAYERS
            if "encoder.layer." in name:

                layer_num = int(
                    name.split("encoder.layer.")[1].split(".")[0]
                )

                if layer_num >= 8:

                    param.requires_grad = True

        # =================================================
        # DROPOUT
        # =================================================

        self.dropout = nn.Dropout(0.2)

    # =====================================================
    # FORWARD PASS
    # =====================================================

    def forward(

        self,

        input_ids,

        attention_mask
    ):

        # =================================================
        # ROBERTA OUTPUTS
        # =================================================

        outputs = self.roberta(

            input_ids=input_ids,

            attention_mask=attention_mask
        )

        # =================================================
        # CLS TOKEN EMBEDDING
        # =================================================

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # Shape:
        # [B, 768]

        # =================================================
        # DROPOUT
        # =================================================

        x = self.dropout(
            cls_embedding
        )

        # =================================================
        # NORMALIZATION
        # =================================================

        x = nn.functional.normalize(

            x,

            dim=1
        )

        # FINAL:
        # [B, 768]

        return x

In [13]:
# =========================================================
# 13. INITIALIZE MODEL
# =========================================================

text_encoder = TextEncoder().to(DEVICE)

print("Text Encoder Initialized")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Text Encoder Initialized


In [14]:
# =========================================================
# 14. TEST FORWARD PASS
# =========================================================

sample_batch = next(iter(train_loader))

input_ids = sample_batch[
    "input_ids"
].to(DEVICE)

attention_mask = sample_batch[
    "attention_mask"
].to(DEVICE)

with torch.no_grad():

    text_output = text_encoder(

        input_ids,

        attention_mask
    )

print("Forward Pass Successful")

print("Embedding Shape:")

print(text_output.shape)

Forward Pass Successful
Embedding Shape:
torch.Size([32, 768])


In [15]:
# =========================================================
# 15. EXTRACT TRAIN EMBEDDINGS
# =========================================================

text_encoder.eval()

train_embeddings = []

train_labels = []

with torch.no_grad():

    for batch in tqdm(train_loader):

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        embeddings = text_encoder(

            input_ids,

            attention_mask
        )

        train_embeddings.append(

            embeddings.cpu().numpy()
        )

        train_labels.extend(

            batch["label"].numpy()
        )

train_embeddings = np.concatenate(
    train_embeddings
)

train_labels = np.array(
    train_labels
)

print("\nTrain Embeddings Shape:")

print(train_embeddings.shape)

100%|██████████| 70/70 [00:49<00:00,  1.41it/s]


Train Embeddings Shape:
(2240, 768)


In [16]:
# =========================================================
# 16. EXTRACT TEST EMBEDDINGS
# =========================================================

test_embeddings = []

test_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        embeddings = text_encoder(

            input_ids,

            attention_mask
        )

        test_embeddings.append(

            embeddings.cpu().numpy()
        )

        test_labels.extend(

            batch["label"].numpy()
        )

test_embeddings = np.concatenate(
    test_embeddings
)

test_labels = np.array(
    test_labels
)

print("\nTest Embeddings Shape:")

print(test_embeddings.shape)

100%|██████████| 18/18 [00:12<00:00,  1.46it/s]


Test Embeddings Shape:
(560, 768)


In [17]:
# =========================================================
# 17. SAVE EMBEDDINGS
# =========================================================

os.makedirs(
    "text_embeddings",
    exist_ok=True
)

np.save(

    "text_embeddings/train_text_embeddings.npy",

    train_embeddings
)

np.save(

    "text_embeddings/train_text_labels.npy",

    train_labels
)

np.save(

    "text_embeddings/test_text_embeddings.npy",

    test_embeddings
)

np.save(

    "text_embeddings/test_text_labels.npy",

    test_labels
)

print("Embeddings Saved")

Embeddings Saved


In [18]:
# =========================================================
# 18. SAVE MODEL
# =========================================================

torch.save(

    text_encoder.state_dict(),

    "text_encoder_roberta.pth"
)

print("Text Encoder Saved")

Text Encoder Saved


## TESTING

In [19]:
# =========================================================
# TEST 1: BASIC FORWARD PASS
# =========================================================

sample_text = "Say the word home"

encoding = tokenizer(

    sample_text,

    return_tensors="pt",

    padding=True,

    truncation=True,

    max_length=CONFIG["MAX_LENGTH"]
)

input_ids = encoding["input_ids"].to(DEVICE)

attention_mask = encoding["attention_mask"].to(DEVICE)

with torch.no_grad():

    embedding = text_encoder(

        input_ids,

        attention_mask
    )

print("Embedding Shape:")

print(embedding.shape)

Embedding Shape:
torch.Size([1, 768])


In [20]:
# =========================================================
# TEST 2: NORMALIZATION CHECK
# =========================================================

norm = torch.norm(
    embedding,
    dim=1
)

print("Embedding Norm:")

print(norm)

Embedding Norm:
tensor([1.])


In [21]:
# =========================================================
# TEST 3: SEMANTIC SIMILARITY
# =========================================================

texts = [

    "Say the word home",

    "Say the word house",

    "Say the word angry"
]

all_embeddings = []

with torch.no_grad():

    for text in texts:

        encoding = tokenizer(

            text,

            return_tensors="pt",

            padding=True,

            truncation=True,

            max_length=CONFIG["MAX_LENGTH"]
        )

        input_ids = encoding["input_ids"].to(DEVICE)

        attention_mask = encoding["attention_mask"].to(DEVICE)

        embedding = text_encoder(

            input_ids,

            attention_mask
        )

        all_embeddings.append(
            embedding
        )

emb1 = all_embeddings[0]

emb2 = all_embeddings[1]

emb3 = all_embeddings[2]

# COSINE SIMILARITIES

sim_home_house = torch.cosine_similarity(

    emb1,

    emb2
)

sim_home_angry = torch.cosine_similarity(

    emb1,

    emb3
)

print("\nSimilarity: home vs house")

print(sim_home_house.item())

print("\nSimilarity: home vs angry")

print(sim_home_angry.item())


Similarity: home vs house
0.9997149705886841

Similarity: home vs angry
0.9996396899223328


In [22]:
# =========================================================
# STRONGER SEMANTIC TEST
# =========================================================

texts = [

    "I feel happy today",

    "I am joyful and excited",

    "I am extremely angry"
]

all_embeddings = []

with torch.no_grad():

    for text in texts:

        encoding = tokenizer(

            text,

            return_tensors="pt",

            padding=True,

            truncation=True,

            max_length=CONFIG["MAX_LENGTH"]
        )

        input_ids = encoding["input_ids"].to(DEVICE)

        attention_mask = encoding["attention_mask"].to(DEVICE)

        embedding = text_encoder(

            input_ids,

            attention_mask
        )

        all_embeddings.append(
            embedding
        )

emb1 = all_embeddings[0]

emb2 = all_embeddings[1]

emb3 = all_embeddings[2]

sim_positive = torch.cosine_similarity(

    emb1,

    emb2
)

sim_opposite = torch.cosine_similarity(

    emb1,

    emb3
)

print("\nHappy vs Joyful:")

print(sim_positive.item())

print("\nHappy vs Angry:")

print(sim_opposite.item())


Happy vs Joyful:
0.9994771480560303

Happy vs Angry:
0.9993652701377869


In [29]:
# ============================================================
# EXPORT FULL TEXT EMBEDDINGS
# ============================================================

full_dataset = TextDataset(
    df
)

print("Dataset Size:")
print(len(full_dataset))

# ============================================================
# DATALOADER
# ============================================================

full_loader = DataLoader(

    full_dataset,

    batch_size=32,

    shuffle=False
)

# ============================================================
# EVALUATION MODE
# ============================================================

text_encoder.eval()

all_embeddings = []

all_labels = []

# ============================================================
# EMBEDDING EXTRACTION
# ============================================================

with torch.no_grad():

    for batch in tqdm(full_loader):

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        labels = batch[
            "label"
        ]

        embeddings = text_encoder(

            input_ids,

            attention_mask
        )

        all_embeddings.append(

            embeddings.cpu().numpy()
        )

        all_labels.extend(

            labels.numpy()
        )

# ============================================================
# CONCATENATE
# ============================================================

all_embeddings = np.concatenate(
    all_embeddings
)

all_labels = np.array(
    all_labels
)

print("\nEmbeddings Shape:")
print(all_embeddings.shape)

print("\nLabels Shape:")
print(all_labels.shape)

# ============================================================
# SAVE
# ============================================================

np.save(

    "full_text_embeddings.npy",

    all_embeddings
)

np.save(

    "full_text_labels.npy",

    all_labels
)

print("\nFull Text Embeddings Saved")

Dataset Size:
2800


100%|██████████| 88/88 [01:01<00:00,  1.43it/s]


Embeddings Shape:
(2800, 768)

Labels Shape:
(2800,)

Full Text Embeddings Saved
